<h2 align="center">AtliQo Bank Credit Card Launch: Phase 2</h2>

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.stats.api as sms
import statsmodels.api as sm
from scipy import stats as st
from matplotlib import pyplot as plt

<h3 style="color:blue" align="center">Pre Campaign</h3>

We want to do a trial run for our new credit card. For this we need to figure out how many customers do we need for our A/B testing. We will form a control and test group. For both of these groups we can figure out number of customers we need based on the statistical power and effect size. 

In [ ]:
alpha = 0.05 # siginifance level 5%
power = 0.8 # 
effect_size = 0.2 # difference between 2 means (control and test group) / sum of thier standard deviation
sms.tt_ind_solve_power( # help us decide to decide the sample size
    effect_size = effect_size,
    alpha = alpha,
    power = power,
    ratio = 1,
    alternative = 'two-sided'
)

393.4056930002526

For effect size 2 we need 393 customers. We have to keep in mind budgeting restrictions while running this campaign hence let us run this for different effect sizes and discuss with business to find out which sample size would be optimal

In [ ]:
# Calculate the required sample size for different effect sizes
effect_sizes = [0.1, 0.2, 0.3, 0.4, 0.5,1]  #  standard deviations greater than control group 

for effect_size in effect_sizes:
    sample_size = sms.tt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1, alternative='two-sided')
    print(f"Effect Size: {effect_size}, Required Sample Size: {int(sample_size)} customers")

Effect Size: 0.1, Required Sample Size: 1570 customers
Effect Size: 0.2, Required Sample Size: 393 customers
Effect Size: 0.3, Required Sample Size: 175 customers
Effect Size: 0.4, Required Sample Size: 99 customers
Effect Size: 0.5, Required Sample Size: 63 customers
Effect Size: 1, Required Sample Size: 16 customers


Based on business requirements, the test should be capable of detecting a minimum 0.4 standard deviation difference between the control and test groups. For the effect size 0.4, we need 100 customers and when we discussed with business, 100 customers is ok in terms of their budgeting constraints for this trail run

<h3 style="color:blue" align="center">Post Campaign</h3>

We will conduct a two-sample z-test to determine whether the test group (new participants) had a higher number of transactions compared to the control group (existing participants).

 - Null Hypothesis (H₀):  
The mean number of transactions in the test group is equal to or less than that of the control group.

 - Alternative Hypothesis (H₁):  
The mean number of transactions in the test group is greater than that of the control group.


### Data Import

In [ ]:
df = pd.read_csv('dataset/avg_transactions_after_campaign.csv')

In [ ]:
print(df.shape)
df.head()

(62, 3)


,campaign_date,control_group_avg_tran,test_group_avg_tran
0,2023-09-10,251.02,401.78
1,2023-09-11,250.77,326.16
2,2023-09-12,248.81,303.92
3,2023-09-13,255.90,363.29
4,2023-09-14,255.86,317.06


In [ ]:
df[
    df['control_group_avg_tran'] >= df['test_group_avg_tran']
   ]

,campaign_date,control_group_avg_tran,test_group_avg_tran
38,2023-10-18,250.94,235.48
55,2023-11-04,252.55,214.06


In [ ]:
df[
    df['control_group_avg_tran'] > df['test_group_avg_tran']
   ].shape[0]/df.shape[0]

0.03225806451612903

In [ ]:
control_mean = df.control_group_avg_tran.mean()
control_std = df.control_group_avg_tran.std()
control_mean,control_std

(np.float64(248.94129032258064), np.float64(9.137869049553624))

In [ ]:
test_mean = df.test_group_avg_tran.mean()
test_std = df.test_group_avg_tran.std()
test_mean,test_std

(np.float64(370.5364516129033), np.float64(63.25415113953286))

In [ ]:
sample_size = df.shape[0]
sample_size

62

In [ ]:
a = (control_std**2)/sample_size
b = (test_std**2)/sample_size
z_score = (test_mean-control_mean)/np.sqrt(b + a)
z_critical = st.norm.ppf(1-alpha)
z_score,z_critical

(np.float64(14.98090307099052), np.float64(1.6448536269514722))

In [ ]:
if z_score > z_critical:
    print('Reject Null Hypothesis')
else :
    print('Fail to reject the Null Hypothesis')

Reject Null Hypothesis


### Test Using p-Value

In [ ]:
# Calculate the p-value corresponding to z score for a right-tailed test
p_value = 1 - st.norm.cdf(z_score)
p_value

np.float64(0.0)

In [ ]:
# Performing Z-test with above considerations
z_statistic, p_value = sm.stats.ztest( df['test_group_avg_tran'],df['control_group_avg_tran'],alternative = 'larger')
z_statistic, p_value

(np.float64(14.980903070990523), np.float64(4.893899020400689e-51))

In [ ]:
if p_value < alpha:
    print('Reject Null Hypothesis')
else :
    print('Fail to reject the Null Hypothesis')

Reject Null Hypothesis
